# bench_gen — Semi-supervised drug response modelling

This notebook focuses on the Genomics of Drug Sensitivity in Cancer (GDSC)
dataset. We benchmark tabular regressors via `BenchmarkRunner`, then reframe the
problem as a binary response classification to demonstrate semi-supervised
training with `SemiSupervisedTabular`.


## 1. Environment
Install Kaggle plus the gradient boosting libraries required by the shared
model registry.


In [ ]:
%%capture
!pip install -q kaggle xgboost lightgbm

## 2. Imports and seeds


In [ ]:
from itertools import cycle
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.ss_models import SemiSupervisedTabular
from pipelines_torch.base import SimplePredictor
from utils.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score
from utils.utils import load_model

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download and parse GDSC
We rely on lightweight heuristics to discover the expression and response CSVs.
Update `expr_path` / `resp_path` below if your local copy uses different names.


In [ ]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_DATASET = "samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc"
DATA_ROOT = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_DATASET,
    local_dir=Path("data_gdsc"),
    description="GDSC dataset",
    kaggle_subdir="genomics-of-drug-sensitivity-in-cancer-gdsc",
)

In [ ]:
expr_path = None
resp_path = None
for csv_path in DATA_ROOT.rglob("*.csv"):
    name = csv_path.name.lower()
    if expr_path is None and ("expression" in name or "rnaseq" in name or "gene" in name):
        expr_path = csv_path
    if resp_path is None and ("response" in name or "ic50" in name):
        resp_path = csv_path

if expr_path is None or resp_path is None:
    raise FileNotFoundError("Could not locate expression/response CSVs. Please set expr_path/resp_path manually.")

expr_df = pd.read_csv(expr_path)
resp_df = pd.read_csv(resp_path)
print(expr_df.shape, resp_df.shape)


In [ ]:
expr_df = expr_df.set_index(expr_df.columns[0])
expr_df = expr_df.select_dtypes(include=[np.number])

if 'cell_line' not in resp_df.columns:
    for candidate in ['CellLine', 'CELL_LINE', 'CELL', 'SAMPLE']:
        if candidate in resp_df.columns:
            resp_df = resp_df.rename(columns={candidate: 'cell_line'})
            break
if 'drug' not in resp_df.columns:
    for candidate in ['Drug', 'DRUG_NAME', 'Compound']:
        if candidate in resp_df.columns:
            resp_df = resp_df.rename(columns={candidate: 'drug'})
            break

ic50_cols = [c for c in resp_df.columns if 'ic50' in c.lower()]
if not ic50_cols:
    raise ValueError("Could not identify an IC50 column in the response file")
resp_df = resp_df.rename(columns={ic50_cols[0]: 'IC50'})
resp_df = resp_df.dropna(subset=['cell_line', 'drug', 'IC50'])
resp_df = resp_df[resp_df['cell_line'].isin(expr_df.index)]

resp_df['log_ic50'] = np.log1p(resp_df['IC50'])

focus_drug = resp_df['drug'].value_counts().idxmax()
resp_subset = resp_df[resp_df['drug'] == focus_drug].copy()

features = expr_df.reindex(resp_subset['cell_line']).to_numpy(dtype=np.float32)
labels_reg = resp_subset['log_ic50'].to_numpy(dtype=np.float32)
print(f"Focused on drug: {focus_drug} with {len(labels_reg)} samples")


## 4. Supervised regression benchmark
We standardise features, split train/validation, and feed them to
`BenchmarkRunner` using the registry regressors.


In [ ]:
scaler = StandardScaler()
X_train, X_val, y_train, y_val = train_test_split(
    features, labels_reg, test_size=0.2, random_state=SEED
)
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

reg_metrics = [mean_squared_error, mean_absolute_error, r2_score]
reg_models = [
    {
        "name": "mlp_regressor",
        "class": MODEL_REGISTRY["mlp_regressor"],
        "params": {"input_dim": X_train.shape[1], "output_dim": 1},
    },
    {
        "name": "xgboost_regressor",
        "class": MODEL_REGISTRY["xgboost_regressor"],
        "params": {"n_estimators": 600, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "random_state": SEED},
    },
    {
        "name": "lightgbm_regressor",
        "class": MODEL_REGISTRY["lightgbm_regressor"],
        "params": {"n_estimators": 800, "learning_rate": 0.05, "num_leaves": 64, "subsample": 0.8, "colsample_bytree": 0.8},
    },
]

runner = BenchmarkRunner(
    model_configs=reg_models,
    augmentations=[None],
    metrics=reg_metrics,
    task_type="regression",
    device="cpu",
    epochs=5,
    batch_size=64,
    use_kfold=False,
    learning_rate=1e-3,
    path_start="bench_gen_supervised",
    random_state=SEED,
)
regression_results = runner.run(X_train, y_train)
regression_results


### Reload checkpoints on the validation split


In [ ]:
def evaluate_regression_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    records = []
    for name in model_names:
        checkpoint = f"{name}_none"
        cfg = next(cfg for cfg in reg_models if cfg["name"] == name)
        try:
            model = load_model(cfg["class"], checkpoint, cfg["params"], path_start="bench_gen_supervised")
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        predictor = SimplePredictor(model, task_type="regression", device="cpu", batch_size=128)
        preds = predictor.predict(X)
        records.append({
            "model": name,
            "mse": float(mean_squared_error(y, preds)),
            "mae": float(mean_absolute_error(y, preds)),
            "r2": float(r2_score(y, preds)),
        })
    return pd.DataFrame.from_records(records)

val_regression_metrics = evaluate_regression_models([m["name"] for m in reg_models], X_val, y_val)
val_regression_metrics


## 5. Optional external validation
Place another drug-response dataset in `data_gdsc/secondary.csv` with matching
cell-line identifiers and gene ordering to reuse the scaler above.


In [ ]:
SECONDARY_PATH = DATA_ROOT / "secondary.csv"
if SECONDARY_PATH.exists():
    secondary_df = pd.read_csv(SECONDARY_PATH)
    if 'cell_line' not in secondary_df.columns or 'log_ic50' not in secondary_df.columns:
        raise ValueError("Secondary dataset must contain 'cell_line' and 'log_ic50' columns")
    secondary_df = secondary_df[secondary_df['cell_line'].isin(expr_df.index)]
    X_secondary = expr_df.reindex(secondary_df['cell_line']).to_numpy(dtype=np.float32)
    X_secondary = scaler.transform(X_secondary)
    y_secondary = secondary_df['log_ic50'].to_numpy(dtype=np.float32)
    secondary_metrics = evaluate_regression_models([m["name"] for m in reg_models], X_secondary, y_secondary)
    secondary_metrics
else:
    print("⚠️ Provide a secondary dataset at data_gdsc/secondary.csv to enable this cell.")


## 6. Semi-supervised binary response classification
To leverage the shared semi-supervised utilities we derive a binary label from
log(IC50) (sensitive vs resistant). Only a fraction of labels are revealed to the
model; the remainder act as unlabeled data for Mean Teacher.


In [ ]:
y_binary = (y_train <= np.median(y_train)).astype(np.int64)
X_train_ssl = X_train.astype(np.float32)
X_val_ssl = X_val.astype(np.float32)
y_val_binary = (y_val <= np.median(y_train)).astype(np.int64)

mask = np.random.default_rng(SEED).random(len(y_binary)) < 0.3
X_labeled = X_train_ssl[mask]
y_labeled = y_binary[mask]
X_unlabeled = X_train_ssl[~mask]

labeled_dataset = TensorDataset(torch.tensor(X_labeled), torch.tensor(y_labeled))
unlabeled_dataset = TensorDataset(torch.tensor(X_unlabeled))
val_dataset = TensorDataset(torch.tensor(X_val_ssl), torch.tensor(y_val_binary))

labeled_loader = DataLoader(labeled_dataset, batch_size=64, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=128, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)


In [ ]:
def train_tabular_ssl(epochs: int = 20, lr: float = 1e-3):
    base_model = MODEL_REGISTRY["mlp_classifier"](input_dim=X_train_ssl.shape[1], num_classes=2)
    model = SemiSupervisedTabular(base_model, num_classes=2, use_mean_teacher=True, noise_std=0.05).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    unlabeled_iter = cycle(unlabeled_loader) if len(unlabeled_loader) > 0 else None
    for epoch in range(epochs):
        if unlabeled_iter is None:
            break
        model.train()
        for xb_l, yb_l in labeled_loader:
            xb_u = next(unlabeled_iter)[0]
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)
            optimizer.zero_grad()
            loss, _ = model.step((xb_l, yb_l), (xb_u, None), epoch)
            loss.backward()
            optimizer.step()
            model.post_step()
        model.eval()
        with torch.no_grad():
            xb, yb = next(iter(val_loader))
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_true = yb.numpy()
        metrics = {
            "accuracy": float(accuracy_score(y_true, preds)),
            "f1_macro": float(f1_score(y_true, preds)),
        }
        history.append(metrics)
        print(f"Epoch {epoch+1}/{epochs} — val F1: {metrics['f1_macro']:.3f}")
    return model, pd.DataFrame(history)


In [ ]:
ssl_tabular_model, ssl_history = train_tabular_ssl(epochs=15, lr=1e-3)
ssl_history


In [ ]:
with torch.no_grad():
    xb, yb = next(iter(val_loader))
    xb = xb.to(DEVICE)
    preds = ssl_tabular_model(xb).argmax(dim=1).cpu().numpy()
    y_true = yb.numpy()
metrics = {
    "accuracy": float(accuracy_score(y_true, preds)),
    "f1_macro": float(f1_score(y_true, preds)),
}
metrics


## 7. Where to go next
- Swap `focus_drug` to analyse other therapeutic compounds.
- Replace the Mean Teacher wrapper with `use_mean_teacher=False` to run pure
  pseudo-labelling.
- Provide a curated secondary cohort (e.g. PRISM / CCLE) to stress-test the
  supervised checkpoints.
